## Reset capa Bronce
Vacía el catálogo STAC indexado y el Volume de datos crudos antes de
empezar a trabajar con una nueva área de interés o rango temporal.

**Destructivo. Correr solo a mano, nunca como Job programado.**

In [0]:
CONFIRM_WIPE = "SI_BORRAR"  # <-- escribí "SI_BORRAR" a mano cada vez que quieras correr esto

if CONFIRM_WIPE != "":
    raise RuntimeError("Confirmación no provista. Editá CONFIRM_WIPE para confirmar.")


In [0]:
import yaml
import os

def find_repo_root(start_dir: str, marker_relative_path: str = "conf/ingestion_config.yaml", max_levels: int = 8) -> str:
    current = start_dir
    for _ in range(max_levels):
        candidate = os.path.join(current, marker_relative_path)
        if os.path.exists(candidate):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            break
        current = parent
    raise FileNotFoundError(
        f"Could not find '{marker_relative_path}' coming from {start_dir}"
    )

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()

notebook_fs_path = f"/Workspace{notebook_path}" if not notebook_path.startswith("/Workspace") else notebook_path
start_dir = os.path.dirname(notebook_fs_path)

repo_root = find_repo_root(start_dir)
config_path = f"{repo_root}/conf/ingestion_config.yaml"

with open(config_path) as f:  
    config = yaml.safe_load(f)

uc = config["unity_catalog"]
volume_root = f"/Volumes/{uc['catalog']}/{uc['bronze_schema']}/{uc['volume_name']}"
stac_table = uc["stac_source_table"]

In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

# --- 1. Vaciar la tabla STAC indexada ---
try:
    spark.sql(f"TRUNCATE TABLE {stac_table}")
    print(f"Tabla STAC vaciada: {stac_table}")
except Exception as e:
    print(f"Tabla STAC no existe todavía (nada que truncar): {e}")


In [0]:
# --- 2. Vaciar el Volume de datos crudos ---
try:
    contents = dbutils.fs.ls(volume_root)
    for item in contents:
        dbutils.fs.rm(item.path, recurse=True)
    print(f"Volume vaciado: {volume_root} ({len(contents)} items borrados)")
except Exception as e:
    print(f"Volume vacío o no existe: {e}")
